In [1]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology  # Needs tqdm library to work
from topologicpy.Dictionary import Dictionary
from topologicpy.Plotly import Plotly
import plotly.graph_objects as go

In [3]:
class ITEDevice:
    """Base class handling for ITE device with Topologic geometry and EIA-310 RU placement."""
    ru_height = 1.75
    ru1_offset_z = 5.0
    ou_height = 1.75
    ou_1_offset_z = 5.0
    
    def __init__(self, start_ru, size_ru, name, make, model):
        self.start_ru = start_ru
        self.size_ru = size_ru
        self.name = name
        self.make = make
        self.model = model
        self.topology = None # Will hold the Topologic Cell
        
    def generate_geometry(self, width, depth):
        """Builds the 3D cell and attaches BIM metadata."""
        eq_h = self.size_ru * self.ru_height
        z_center = self.ru1_offset_z + ((self.start_ru - 1) * self.ru_height) + (eq_h / 2.0)
        
        origin = Vertex.ByCoordinates(0, 0, z_center)
        cell = Cell.Box(origin=origin, width=width, length=depth, height=eq_h)
        
        # Inject universal metadata
        keys = ["Device_Name", "Make", "Model", "Starting_RU"]
        values = [self.name, self.make, self.model, self.start_ru]
        
        self.topology = Topology.SetDictionary(cell, Dictionary.ByKeysValues(keys, values))
        
        return self.topology


class ITEPassiveDevice(ITEDevice):
    """Base class for passive ITE devices."""
    
    def __init__(self):

        super().__init__(start_unit, size_units=4)


class ITEActiveDevice(ITEDevice):

    def __init__(self, power_supplies):

        return super().generate_geometry(width=19, depth=18)

# ==============================================================================
# 2. Subclasses (Inheritance)
# ==============================================================================
class ServerDevice(ITEActiveDevice):
    """Subclass for heavy compute nodes. Presets size and deeper chassis."""
    def __init__(self, start_ru, name):
        # Call the parent constructor with predetermined compute specs
        super().__init__(start_ru=start_ru, size_ru=2, name=name, make="NVIDIA", model="HGX")
        
    def generate_geometry(self):
        # Compute nodes are typically deeper (e.g., 36 inches)
        
        return super().generate_geometry(width=19.0, depth=36.0)


class NetworkDevice(ITEActiveDevice):
    """Subclass for network switches or routers. Presets 1U size and shallower chassis."""
    def __init__(self, start_ru, name):
        # Call the parent constructor with predetermined network specs
        super().__init__(start_ru=start_ru, size_ru=1, name=name, make="Whitebox", model="OcNOS-400G")
        
    def generate_geometry(self):
        # Switches are typically much shallower (e.g., 18 inches)
        
        return super().generate_geometry(width=19.0, depth=18.0)

In [8]:
# ==============================================================================
# 3. Clean Instantiation
# ==============================================================================
# Your main script becomes incredibly clean and readable
node_1 = ServerDevice(start_ru=10, name="AI_Compute_01")
node_2 = ServerDevice(start_ru=13, name="AI_Compute_02")
spine  = NetworkDevice(start_ru=42, name="Spine_Switch")

# Generate the Topologic cells
equipment_cells = [
    node_1.generate_geometry(), 
    node_2.generate_geometry(), 
    spine.generate_geometry()
]

Topology.Show(equipment_cells)

TypeError: ITEActiveDevice.__init__() got an unexpected keyword argument 'start_ru'

In [6]:
equipment_cells

NameError: name 'equipment_cells' is not defined

In [ ]:
Topology.Show(equipment_cells[0])